In this Notebook, we create an animation for some of the FSSP solvers

In [ ]:
# standard preamble for the Notebooks I use
import torch as tc
import igraph as ig
import numpy as np
from matplotlib import pyplot as plt
from matplotlib import rcParams

# Enable LaTeX and set Times New Roman as the font
rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Times New Roman"],
    "text.latex.preamble": r"\usepackage{amsmath}"  # Optional: Use LaTeX packages
})

from tqdm import tqdm

import sys, os
current_dir = os.getcwd()
parent_dir = os.path.abspath(os.path.join(current_dir, '..'))
sys.path.append(parent_dir)

# compact saving of data
import h5py

from src.automata import LLNA
from src.simulation import *
from src.rules import binary_indices, return_equivalent_rule, get_nonequiv_rules, return_life_like_dict
from src.networks import create_2d_torus_lattice, watts_strogatz_rewire
from src.analysis import median_and_percentiles_over_ensemble, mean_field_dens_propagation, derrida_map_analytical, hamming_weight, boolean_sens, mean_field_slope

%load_ext autoreload
%autoreload 2

# Find all rules that could work

There are two strict criteria for a rule that could solve the FSSP
1. It's self-equivalent (not sure whether that is true though)
2. It "wants" to move to the extreme states, in other words
$$
\begin{equation*}
\left\{
\begin{array}{ll}
\langle \rho^{t+1} \rangle \geq 1 - \rho^t & \text{if } \rho^t \in ]0, \frac{1}{2}], \\[6pt]
\langle \rho^{t+1} \rangle < 1 - \rho^t & \text{if } \rho^t \in ]\frac{1}{2}, 1[,
\end{array}
\right.
\end{equation*}
$$
which requires an S-shaped curve.

In [ ]:
resolution = 9
degree = 8

# self-equivalence
nonequiv_rule_list_name = f"../data/rule_tables/all_nonequiv_res{resolution}_rules.npy"
self_equiv_rule_list_name = f"../data/rule_tables/all_self_equiv_res{resolution}_rules.npy"

nonequiv_rule_list = np.load(nonequiv_rule_list_name)
self_equiv_rule_list = np.load(self_equiv_rule_list_name)

len(self_equiv_rule_list)

In [ ]:
rho0_number = 1001
rho0_array = np.linspace(0., 1., rho0_number)

def test_fssp_requirement(mfdp_array, precision=8):
    mfdp_array = np.round(np.asarray(mfdp_array, dtype=float), precision)
    n = mfdp_array.size
    if n % 2 == 0:
        raise ValueError("Length of mfdp_array should be odd.")

    rho = np.round(np.linspace(0.0, 1.0, n), precision)

    # Masks per the spec: (0, 1/2] and (1/2, 1)
    left  = (rho > 0.0) & (rho <= 0.5)
    right = (rho > 0.5) & (rho < 1.0)

    cond_left = mfdp_array[left]  >= 1 - rho[left]
    cond_right = mfdp_array[right] <  1 - rho[right]

    bool_left  = np.all(cond_left)
    bool_right = np.all(cond_right)

    return bool(bool_left or bool_right)

# find unstable subset
mfdp_array_stack = np.empty((0, rho0_number))
unstable_illegal_self_equiv_rule_list = []
for beta, sigma in self_equiv_rule_list:
    B_set = binary_indices(beta)
    S_set = binary_indices(sigma)
    mfdp_array = mean_field_dens_propagation(resolution, B_set, S_set, rho0_array, degree, iso=True)
    # print(mfdp_array)
    if test_fssp_requirement(mfdp_array):
        unstable_illegal_self_equiv_rule_list.append((beta, sigma))
        mfdp_array_stack = np.vstack((mfdp_array_stack, mfdp_array))

number_candidates = len(unstable_illegal_self_equiv_rule_list)
print(f"Number of candidates: {number_candidates}")

In [ ]:
beta, sigma = unstable_illegal_self_equiv_rule_list[np.random.randint(number_candidates)]
B_set = binary_indices(beta)
S_set = binary_indices(sigma)

rho0_array = np.linspace(0,1,101)
degree = 8
rho1_array = mean_field_dens_propagation(resolution, B_set, S_set, rho0_array, degree)

rho_star = 0.5
slope = mean_field_slope(resolution, B_set, S_set, degree, rho_star=rho_star)
intercept = 0.5*(1-slope)

plt.plot(rho0_array, rho1_array, label=f"Degree {degree}", lw=3)
plt.plot([0,1], [0,1], lw=1, ls='--', color='k')
plt.plot([0,1], [1,0], lw=1, ls='--', color='k')
plt.plot([0, 1], [intercept, slope+intercept], lw=1, ls=':', color='k', label="Tangent at central equilibrium")
plt.legend()

plt.xlim([0,1])
plt.ylim([0,1])

plt.title(f"$\\beta = {beta}$, $\\sigma = {sigma}$")

plt.gca().set_aspect('equal', adjustable='box')

# Make an animation for three different topologies

In [ ]:
# Parameters
width = 100                  # Grid size (L x L) WATCH OUT with values, becomes intensive fast
num_nodes = width**2
rewiring_prob = 0.2     # Rewiring probability
degree = 8
num_edges = num_nodes*degree//2

# Create toroidal lattice and rewire.
# REWIRING CAN TAKE A WHILE!
lattice_graph = create_2d_torus_lattice(width, degree=degree)
small_world_graph = watts_strogatz_rewire(lattice_graph, rewiring_prob)
random_graph = ig.Graph.Erdos_Renyi(n=num_nodes, m=num_edges) # watts_strogatz_rewire(lattice_graph, 1.)

# find edges
lattice_graph.to_directed()
lattice_edges = tc.tensor(lattice_graph.get_edgelist()).T
lattice_graph.to_undirected()

small_world_graph.to_directed()
small_world_edges = tc.tensor(small_world_graph.get_edgelist()).T
small_world_graph.to_undirected()

random_graph.to_directed()
random_edges = tc.tensor(random_graph.get_edgelist()).T
random_graph.to_undirected()

# save graphs and edges in list
graphs = [lattice_graph, small_world_graph, random_graph]
graph_names = ["Lattice", f"Small World ($p={rewiring_prob}$)", "Random"]
edges = [lattice_edges, small_world_edges, random_edges]

In [ ]:
beta, sigma = (23, 47) # life_like_dict["anneal"] # (488, 464) # (226, 369) # (232, 465) # candidate_rules[1]
B_set = binary_indices(beta)
S_set = binary_indices(sigma)
model = LLNA(resolution, B_set, S_set, iso=True)
model.diagram()

In [ ]:
def init_config_with_dens(N, dens):
    # defines a random initial configuration with a fixed state density
    s0 = np.zeros(N, dtype=int)
    s0[:np.round(dens*N).astype(int)] = 1.
    np.random.shuffle(s0)
    return s0

T = 100
num_config = 1
init_dens = 0.5

init_configs = np.array([init_config_with_dens(num_nodes, init_dens) for _ in range(num_config)])
# init_defects = np.array([init_config_with_dens(L*L, 1/L/L) for _ in range(num_config)])
# init_configs_defect = (init_configs + init_defects) % 2
# run simulation over a number of time steps
configs = model.forward(random_edges, tc.tensor(init_configs), T=T).numpy().astype(int) # lattice_eges, small_world_edges, random_edges
# configs_defect = model.forward(lattice_edges, tc.tensor(init_configs_defect), T=T).numpy().astype(int)
# defects = (configs + configs_defect) % 2

# put back in L x L shape 
configs = configs.reshape((num_config, T+1, width, width))
# defects = defects.reshape((num_config, T+1, L, L))

In [ ]:
# animations
import matplotlib.animation as animation
from IPython.display import HTML

def make_animation_object(grids, title=None):
    # Set up figure
    fig, ax = plt.subplots(figsize=(5, 5))
    im = ax.imshow(grids[0])
    # Update function for animation
    def update(frame):
        im.set_array(grids[frame])
        ax.set_title(title, fontsize=20)
        ax.set_xticks([]); ax.set_yticks([])
    # Create animation
    ani = animation.FuncAnimation(fig, update, frames=T+1, interval=100)
    plt.close()
    return ani

In [ ]:
# show an animation. small world blijft krioelen voor day&night! random ook.
# threshold model wordt heel saai voor small world en voor random network, dus er is wel degelijk iets te zeggen voor het annealed model

graph_name = graph_names[0]
example_idx = 0 # np.random.randint(num_config)
ani = make_animation_object(configs[example_idx], title=f"{model.__str__(latex=True)} on graph `{graph_name}'")
# ani = make_animation_object(defects[example_idx])

# show inline
HTML(ani.to_jshtml())

In [ ]:
# show an animation. small world blijft krioelen voor day&night! random ook.
# threshold model wordt heel saai voor small world en voor random network, dus er is wel degelijk iets te zeggen voor het annealed model

graph_name = graph_names[1]
example_idx = 0 # np.random.randint(num_config)
ani = make_animation_object(configs[example_idx], title=f"{model.__str__(latex=True)} on graph `{graph_name}'")
# ani = make_animation_object(defects[example_idx])

# show inline
HTML(ani.to_jshtml())

In [ ]:
# show an animation. small world blijft krioelen voor day&night! random ook.
# threshold model wordt heel saai voor small world en voor random network, dus er is wel degelijk iets te zeggen voor het annealed model

graph_name = graph_names[2]
example_idx = 0 # np.random.randint(num_config)
ani = make_animation_object(configs[example_idx], title=f"{model.__str__(latex=True)} on graph `{graph_name}'")
# ani = make_animation_object(defects[example_idx])

# show inline
HTML(ani.to_jshtml())

In [ ]:
# save some GIFs for future reference

graph_name = graph_names[0]

SAVEGIFS = True
life_like_name = 'fssp'

if SAVEGIFS:
    for i, config in tqdm(enumerate(configs), total=configs.shape[0]):
        savename = f"animation_llca_{life_like_name}_{width}x{width}_T{T}_sample{i}.gif"

        # example_idx = np.random.randint(num_config)
        ani = make_animation_object(config, title=f"{model.__str__(latex=True)} on graph `{graph_name}'")
        # ani = make_animation_object(defects[example_idx])

        # Save animations as GIFs
        loc = '../figures/gifs/'
        ani.save(loc+savename, writer='pillow', fps=24)  # Using Pillow

        # show inline
        # HTML(ani.to_jshtml())